# Laboratório — Divergência KL e informação mútua

## Goal

Calcular KL, Jensen–Shannon, informação mútua (MI) e informação mútua pontual (PMI); verificar identidades; observar assimetria, suporte, dependência não linear, interação XOR, viés em amostras finitas e desigualdade de processamento.

Este notebook é o laboratório da [Aula 23](../aulas/23-kl-informacao-mutua.md).

## Setup

**Ambiente de referência:** Python 3.11+, NumPy 2+, pandas 2+, Matplotlib 3.8+ e SciPy 1.13+.

No Colab, se necessário:

```python
%pip install "numpy>=2.0" "pandas>=2.0" "matplotlib>=3.8" "scipy>=1.13"
```

O notebook não baixa dados. Todos os experimentos são sintéticos e usam seed fixa.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.stats import entropy as scipy_entropy

SEED = 20260908
rng = np.random.default_rng(SEED)

print(f"Python {platform.python_version()}")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")
print(f"Matplotlib {matplotlib.__version__} | SciPy {scipy.__version__}")
print(f"Seed: {SEED}")

## Steps

### 1. Implemente as quantidades com validação

As funções abaixo tratam explicitamente probabilidades zero. Por padrão, o logaritmo natural produz nats; use `base=2` para bits.

In [ ]:
def validar_probabilidades(valores, nome="p"):
    p = np.asarray(valores, dtype=float)
    if p.ndim < 1 or p.size == 0:
        raise ValueError(f"{nome} deve ter ao menos um valor")
    if not np.all(np.isfinite(p)) or np.any(p < 0):
        raise ValueError(f"{nome} contém valor inválido")
    if not np.isclose(p.sum(), 1.0, atol=1e-12):
        raise ValueError(f"{nome} deve somar 1; soma={p.sum()}")
    return p


def entropia(p, base=np.e):
    p = validar_probabilidades(p)
    positivos = p > 0
    return float(-np.sum(p[positivos] * np.log(p[positivos])) / np.log(base))


def cross_entropy(p, q, base=np.e):
    p = validar_probabilidades(p, "p")
    q = validar_probabilidades(q, "q")
    if p.shape != q.shape:
        raise ValueError("p e q devem ter o mesmo shape")
    if np.any((p > 0) & (q == 0)):
        return np.inf
    positivos = p > 0
    return float(-np.sum(p[positivos] * np.log(q[positivos])) / np.log(base))


def kl_divergence(p, q, base=np.e):
    p = validar_probabilidades(p, "p")
    q = validar_probabilidades(q, "q")
    if p.shape != q.shape:
        raise ValueError("p e q devem ter o mesmo shape")
    if np.any((p > 0) & (q == 0)):
        return np.inf
    positivos = p > 0
    return float(np.sum(p[positivos] * np.log(p[positivos] / q[positivos])) / np.log(base))


def mutual_information(joint, base=np.e):
    joint = validar_probabilidades(joint, "joint")
    if joint.ndim != 2:
        raise ValueError("joint deve ser uma matriz 2D")
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    independente = px @ py
    return kl_divergence(joint.ravel(), independente.ravel(), base=base)


def pmi_matrix(joint, base=np.e):
    joint = validar_probabilidades(joint, "joint")
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    denominador = px @ py
    resultado = np.full_like(joint, -np.inf)
    positivos = joint > 0
    resultado[positivos] = np.log(joint[positivos] / denominador[positivos]) / np.log(base)
    return resultado

### 2. Verifique assimetria e direção

`P` representa a distribuição de referência; `Q`, a aproximação. Inverter os argumentos muda os pesos do valor esperado.

In [ ]:
p = np.array([0.9, 0.1])
q = np.array([0.5, 0.5])
kl_pq = kl_divergence(p, q)
kl_qp = kl_divergence(q, p)

comparacao_kl = pd.DataFrame({
    "direção": ["KL(P || Q)", "KL(Q || P)"],
    "nats": [kl_pq, kl_qp],
    "bits": [kl_divergence(p, q, 2), kl_divergence(q, p, 2)],
})
print(comparacao_kl.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

### 3. Decomponha cross-entropy

Testamos numericamente a identidade `CE(P,Q) = H(P) + KL(P||Q)` e reconciliamos nossa implementação com `scipy.stats.entropy`.

In [ ]:
h_p = entropia(p)
ce_pq = cross_entropy(p, q)
decomposicao = h_p + kl_pq
kl_scipy = scipy_entropy(p, q)

print(f"H(P)             = {h_p:.9f} nat")
print(f"KL(P || Q)       = {kl_pq:.9f} nat")
print(f"H(P) + KL        = {decomposicao:.9f} nat")
print(f"Cross-entropy    = {ce_pq:.9f} nat")
print(f"SciPy KL         = {kl_scipy:.9f} nat")

### 4. Observe incompatibilidade de suporte

Se `P` considera um resultado possível e `Q` atribui zero a ele, a KL é infinita. A suavização produz um valor finito, mas o resultado depende do parâmetro escolhido.

In [ ]:
q_sem_cobertura = np.array([1.0, 0.0])
kl_infinita = kl_divergence(p, q_sem_cobertura)

linhas = []
for epsilon in [1e-1, 1e-3, 1e-6, 1e-12]:
    q_suave = q_sem_cobertura + epsilon
    q_suave = q_suave / q_suave.sum()
    linhas.append({"epsilon": epsilon, "KL(P || Q_suave)": kl_divergence(p, q_suave)})

print("KL sem cobertura:", kl_infinita)
print(pd.DataFrame(linhas).to_string(index=False, float_format=lambda x: f"{x:.9g}"))

### 5. Compare com Jensen–Shannon

A divergência Jensen–Shannon usa a mistura entre as distribuições. Com log base 2 e pesos iguais, fica entre 0 e 1 bit.

In [ ]:
def js_divergence(p, q, base=2):
    p = validar_probabilidades(p, "p")
    q = validar_probabilidades(q, "q")
    mistura = (p + q) / 2
    return 0.5 * kl_divergence(p, mistura, base) + 0.5 * kl_divergence(q, mistura, base)


js_pq = js_divergence(p, q)
js_qp = js_divergence(q, p)
print(f"JS(P,Q)={js_pq:.9f} bit")
print(f"JS(Q,P)={js_qp:.9f} bit")

### 6. Calcule MI e PMI em uma tabela 2 × 2

As marginais são uniformes, mas a conjunta concentra massa na diagonal. A MI compara a conjunta com o produto das marginais.

In [ ]:
joint = np.array([[0.4, 0.1],
                  [0.1, 0.4]])
mi_nats = mutual_information(joint)
mi_bits = mutual_information(joint, base=2)
pmi_bits = pmi_matrix(joint, base=2)

px = joint.sum(axis=1)
py = joint.sum(axis=0)
h_x = entropia(px, base=2)
h_y = entropia(py, base=2)
h_xy = entropia(joint.ravel(), base=2)

print("Conjunta:")
print(pd.DataFrame(joint, index=["X=0", "X=1"], columns=["Y=0", "Y=1"]))
print()
print("PMI por célula (bits):")
print(pd.DataFrame(pmi_bits, index=["X=0", "X=1"], columns=["Y=0", "Y=1"]).round(6))
print()
print(f"MI={mi_nats:.9f} nat = {mi_bits:.9f} bit")
print(f"H(X)+H(Y)-H(X,Y)={h_x + h_y - h_xy:.9f} bit")

### 7. Visualize contribuição e PMI

PMI pode ser negativa em células específicas. A MI é a média ponderada dessas contribuições e permanece não negativa.

In [ ]:
contribuicoes = joint * pmi_bits
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))

for ax, matriz, titulo, formato in [
    (axes[0], pmi_bits, "PMI por par (bits)", ".3f"),
    (axes[1], contribuicoes, "Contribuição para MI (bits)", ".3f"),
]:
    imagem = ax.imshow(matriz, cmap="coolwarm")
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            ax.text(j, i, format(matriz[i, j], formato), ha="center", va="center")
    ax.set(xticks=[0, 1], yticks=[0, 1], xlabel="Y", ylabel="X", title=titulo)
    fig.colorbar(imagem, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()
print(f"Soma das contribuições={contribuicoes.sum():.9f} bit")

### 8. Detecte dependência não linear

Geramos `Y = X² + ruído`. A correlação linear fica próxima de zero por simetria, mas MI discretizada permanece positiva. Os bins são ajustados apenas neste conjunto didático; em ML, ajuste-os dentro do treino.

In [ ]:
def mi_empirica_discreta(x, y, base=2):
    x = np.asarray(x)
    y = np.asarray(y)
    if x.shape != y.shape or x.ndim != 1:
        raise ValueError("x e y devem ser vetores alinhados")
    _, xi = np.unique(x, return_inverse=True)
    _, yi = np.unique(y, return_inverse=True)
    contagens = np.zeros((xi.max() + 1, yi.max() + 1), dtype=float)
    np.add.at(contagens, (xi, yi), 1)
    return mutual_information(contagens / contagens.sum(), base=base)


n = 8_000
x_cont = rng.uniform(-1, 1, size=n)
y_cont = x_cont**2 + rng.normal(0, 0.08, size=n)
correlacao = np.corrcoef(x_cont, y_cont)[0, 1]

quantis_x = np.unique(np.quantile(x_cont, np.linspace(0, 1, 11)))
quantis_y = np.unique(np.quantile(y_cont, np.linspace(0, 1, 11)))
x_bins = np.digitize(x_cont, quantis_x[1:-1])
y_bins = np.digitize(y_cont, quantis_y[1:-1])
mi_nao_linear = mi_empirica_discreta(x_bins, y_bins)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x_cont[::8], y_cont[::8], s=8, alpha=0.35)
ax.set(xlabel="X", ylabel="Y", title="Dependência não linear com correlação próxima de zero")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Correlação de Pearson={correlacao:.6f}")
print(f"MI discretizada={mi_nao_linear:.6f} bits")

### 9. Veja uma interação XOR invisível marginalmente

Cada entrada isolada é independente do alvo. O par `(X1, X2)` determina `Y` perfeitamente.

In [ ]:
repeticoes = 2_000
x1 = rng.integers(0, 2, size=repeticoes)
x2 = rng.integers(0, 2, size=repeticoes)
y_xor = x1 ^ x2
par = 2 * x1 + x2

mi_x1 = mi_empirica_discreta(x1, y_xor)
mi_x2 = mi_empirica_discreta(x2, y_xor)
mi_par = mi_empirica_discreta(par, y_xor)

print(f"I(X1;Y)={mi_x1:.6f} bit")
print(f"I(X2;Y)={mi_x2:.6f} bit")
print(f"I((X1,X2);Y)={mi_par:.6f} bit")

### 10. Meça o viés *plug-in* sob independência

Com muitas categorias e poucos dados, a MI empírica é positiva mesmo quando as variáveis foram geradas independentemente. A permutação fornece um baseline sob a hipótese nula com as marginais observadas.

In [ ]:
n_pequeno = 120
categorias = 12
x_ind = rng.integers(0, categorias, size=n_pequeno)
y_ind = rng.integers(0, categorias, size=n_pequeno)
mi_observada = mi_empirica_discreta(x_ind, y_ind)

mi_permutadas = np.array([
    mi_empirica_discreta(x_ind, rng.permutation(y_ind))
    for _ in range(1_000)
])
p_permutacao = (1 + np.count_nonzero(mi_permutadas >= mi_observada)) / (len(mi_permutadas) + 1)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(mi_permutadas, bins=30, alpha=0.8, label="MI após permutar Y")
ax.axvline(mi_observada, color="crimson", linewidth=2, label="MI observada")
ax.set(xlabel="MI plug-in (bits)", ylabel="Frequência", title="Baseline nulo para variáveis independentes")
ax.legend()
plt.tight_layout()
plt.show()

print(f"MI observada={mi_observada:.6f} bit")
print(f"Média sob permutação={mi_permutadas.mean():.6f} bit")
print(f"p-value de permutação={p_permutacao:.6f}")

### 11. Verifique a desigualdade de processamento

Criamos a cadeia `X → Y → Z`: primeiro há 10% de inversões; depois, mais 20%. O segundo canal adiciona ruído sem observar `X` diretamente.

In [ ]:
n_canal = 200_000
x_origem = rng.integers(0, 2, size=n_canal)
y_canal = x_origem ^ (rng.random(n_canal) < 0.10)
z_canal = y_canal ^ (rng.random(n_canal) < 0.20)

mi_xy = mi_empirica_discreta(x_origem, y_canal)
mi_xz = mi_empirica_discreta(x_origem, z_canal)

print(f"I(X;Y)={mi_xy:.6f} bit")
print(f"I(X;Z)={mi_xz:.6f} bit")
print(f"Informação perdida no segundo canal={mi_xy-mi_xz:.6f} bit")

## Checks

As asserções abaixo reconciliam resultados exatos, propriedades teóricas e saídas simuladas. Elas também impedem que uma mudança silenciosa nas funções altere as conclusões.

In [ ]:
assert np.isclose(kl_pq, 0.3680642071684971)
assert np.isclose(kl_qp, 0.5108256237659907)
assert not np.isclose(kl_pq, kl_qp)
assert np.isclose(ce_pq, decomposicao)
assert np.isclose(kl_pq, kl_scipy)
assert np.isinf(kl_infinita)
assert np.isclose(js_pq, js_qp) and 0 <= js_pq <= 1
assert np.isclose(mi_nats, 0.19274475702175753)
assert np.isclose(mi_bits, h_x + h_y - h_xy)
assert np.isclose(contribuicoes.sum(), mi_bits)
assert abs(correlacao) < 0.05 and mi_nao_linear > 0.5
assert mi_x1 < 0.01 and mi_x2 < 0.01 and mi_par > 0.99
assert mi_permutadas.mean() > 0
assert mi_xz < mi_xy

print("Todas as verificações foram aprovadas.")
print(f"KL(P||Q)={kl_pq:.6f}; KL(Q||P)={kl_qp:.6f} nat")
print(f"MI da tabela={mi_bits:.6f} bit")
print(f"Não linear: r={correlacao:.6f}; MI={mi_nao_linear:.6f} bits")
print(f"XOR: MI marginal máxima={max(mi_x1, mi_x2):.6f}; MI conjunta={mi_par:.6f} bit")
print(f"Viés nulo médio={mi_permutadas.mean():.6f} bit; p={p_permutacao:.6f}")
print(f"Processamento: I(X;Y)={mi_xy:.6f} > I(X;Z)={mi_xz:.6f} bit")

## Next Steps

- Troque a matriz conjunta e confirme manualmente as marginais.
- Aumente `n_pequeno` e observe a redução do baseline de MI sob independência.
- Varie a quantidade de categorias e a discretização da relação não linear.
- Em um projeto real, coloque discretização e seleção de atributos dentro de cada dobra de treino.
- Siga para a [Aula 24 — Capstone P3](../aulas/24-capstone-experimento-estatistico-honesto.md) e transforme uma pergunta em experimento reproduzível.